# Chapter 10: Data Analysis Project — Complete Workflow

In this capstone chapter, we bring everything together in a real-world  
data analysis workflow using a **synthetic e-commerce dataset**.

## Workflow Steps

1. 📦 Generate synthetic data (customers, products, orders)
2. 💾 Save to CSV
3. 📂 Load & inspect data
4. 🧹 Clean: missing values, dtypes, duplicates
5. 🔗 Merge all tables
6. 🔍 EDA: top customers, best products, revenue trends
7. 📊 GroupBy analysis: by category, region, month
8. 📈 Time series: monthly revenue, rolling averages
9. 📋 Pivot tables & cross-tabulations
10. 📝 Summary report

In [ ]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Step 1: Generate Synthetic E-Commerce Data

In [ ]:
# --- Customers ---
n_customers = 500
regions = ['North', 'South', 'East', 'West', 'Central']
cities = {
    'North': ['Chicago', 'Detroit', 'Minneapolis'],
    'South': ['Houston', 'Miami', 'Atlanta'],
    'East': ['New York', 'Boston', 'Philadelphia'],
    'West': ['Los Angeles', 'Seattle', 'Denver'],
    'Central': ['Dallas', 'Kansas City', 'St Louis'],
}

cust_regions = np.random.choice(regions, n_customers)
cust_cities = [np.random.choice(cities[r]) for r in cust_regions]

first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer',
               'Michael', 'Linda', 'David', 'Elizabeth', 'William', 'Barbara',
               'Richard', 'Susan', 'Joseph', 'Jessica', 'Thomas', 'Sarah',
               'Charles', 'Karen', 'Emma', 'Oliver', 'Ava', 'Liam', 'Sophia']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia',
              'Miller', 'Davis', 'Rodriguez', 'Martinez', 'Wilson', 'Anderson',
              'Taylor', 'Thomas', 'Moore', 'Jackson', 'Martin', 'Lee',
              'Thompson', 'White', 'Clark', 'Lewis', 'Hall', 'Young', 'King']

customers = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'first_name': np.random.choice(first_names, n_customers),
    'last_name': np.random.choice(last_names, n_customers),
    'email': [f'user{i}@email.com' for i in range(1, n_customers + 1)],
    'region': cust_regions,
    'city': cust_cities,
    'join_date': pd.date_range('2020-01-01', periods=n_customers, freq='D').astype(str),
})

# Inject some missing values
mask_email = np.random.random(n_customers) < 0.05
customers.loc[mask_email, 'email'] = None

mask_city = np.random.random(n_customers) < 0.03
customers.loc[mask_city, 'city'] = None

print(f'Customers: {len(customers)} rows')
customers.head()

In [ ]:
# --- Products ---
products = pd.DataFrame({
    'product_id': range(1, 21),
    'product_name': [
        'Laptop Pro', 'Wireless Mouse', 'Mechanical Keyboard', 'Monitor 27"',
        'USB-C Hub', 'Webcam HD', 'Headphones', 'Desk Lamp',
        'Phone Stand', 'Cable Organizer', 'External SSD', 'Power Bank',
        'Tablet Case', 'Screen Protector', 'HDMI Cable', 'Mouse Pad',
        'Speaker', 'Chair Mat', 'Desk Organizer', 'Notebook Stand'
    ],
    'category': [
        'Computers', 'Accessories', 'Accessories', 'Computers',
        'Accessories', 'Electronics', 'Electronics', 'Office',
        'Accessories', 'Office', 'Storage', 'Electronics',
        'Accessories', 'Accessories', 'Accessories', 'Accessories',
        'Electronics', 'Office', 'Office', 'Office'
    ],
    'price': [
        1299.99, 29.99, 89.99, 449.99,
        49.99, 79.99, 149.99, 34.99,
        19.99, 14.99, 129.99, 39.99,
        24.99, 9.99, 12.99, 19.99,
        99.99, 44.99, 29.99, 59.99
    ],
})

print(f'Products: {len(products)} rows')
products

In [ ]:
# --- Orders ---
n_orders = 5000
order_dates = pd.date_range('2023-01-01', '2024-12-31', freq='h')
order_dates = np.random.choice(order_dates, n_orders, replace=True)

orders = pd.DataFrame({
    'order_id': range(10001, 10001 + n_orders),
    'customer_id': np.random.choice(range(1, n_customers + 1), n_orders),
    'product_id': np.random.choice(range(1, 21), n_orders),
    'quantity': np.random.choice([1, 1, 1, 2, 2, 3, 4, 5], n_orders),
    'order_date': sorted(order_dates),
})

# Inject some issues
# Duplicate orders (0.5%)
dup_indices = np.random.choice(n_orders, 25, replace=False)
duplicates = orders.iloc[dup_indices].copy()
orders = pd.concat([orders, duplicates], ignore_index=True)

# Missing quantities (1%)
mask_qty = np.random.random(len(orders)) < 0.01
orders.loc[mask_qty, 'quantity'] = np.nan

# Store dates as strings (to practice parsing)
orders['order_date'] = orders['order_date'].astype(str)

print(f'Orders: {len(orders)} rows (including duplicates and issues)')
orders.head()

## Step 2: Save to CSV

In [ ]:
os.makedirs('data', exist_ok=True)

customers.to_csv('data/customers.csv', index=False)
products.to_csv('data/products.csv', index=False)
orders.to_csv('data/orders.csv', index=False)

print('✅ Data saved to data/ directory:')
for f in os.listdir('data'):
    size = os.path.getsize(f'data/{f}') / 1024
    print(f'   {f}: {size:.1f} KB')

## Step 3: Load & Inspect Data

In [ ]:
# Load with appropriate settings
customers = pd.read_csv('data/customers.csv', parse_dates=['join_date'])
products = pd.read_csv('data/products.csv')
orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])

print('=== Customers ===')
print(f'Shape: {customers.shape}')
print(customers.dtypes)
print()

print('=== Products ===')
print(f'Shape: {products.shape}')
print(products.dtypes)
print()

print('=== Orders ===')
print(f'Shape: {orders.shape}')
print(orders.dtypes)

In [ ]:
# Check for missing values across all tables
for name, df in [('Customers', customers), ('Products', products), ('Orders', orders)]:
    missing = df.isna().sum()
    if missing.any():
        print(f'\n{name} — Missing values:')
        print(missing[missing > 0])

## Step 4: Data Cleaning

### 4a. Handle Missing Values

In [ ]:
# Customers: fill missing emails with placeholder
customers['email'] = customers['email'].fillna('unknown@email.com')

# Customers: fill missing cities with 'Unknown'
customers['city'] = customers['city'].fillna('Unknown')

# Orders: fill missing quantities with median
orders['quantity'] = orders['quantity'].fillna(orders['quantity'].median())

# Verify
print('Missing values after cleaning:')
print(f'  Customers: {customers.isna().sum().sum()}')
print(f'  Products:  {products.isna().sum().sum()}')
print(f'  Orders:    {orders.isna().sum().sum()}')

### 4b. Fix Data Types

In [ ]:
# Ensure quantity is integer
orders['quantity'] = orders['quantity'].astype(int)

# Convert category to categorical for memory efficiency
products['category'] = products['category'].astype('category')
customers['region'] = customers['region'].astype('category')

print('Dtypes after fixing:')
print(orders.dtypes)

### 4c. Remove Duplicates

In [ ]:
print(f'Orders before dedup: {len(orders)}')
n_dups = orders.duplicated().sum()
print(f'Duplicate orders found: {n_dups}')

orders = orders.drop_duplicates().reset_index(drop=True)
print(f'Orders after dedup:  {len(orders)}')

## Step 5: Merge All Tables

In [ ]:
# Merge orders with customers
df = pd.merge(orders, customers, on='customer_id', how='left')

# Merge with products
df = pd.merge(df, products, on='product_id', how='left')

# Calculate revenue
df['revenue'] = df['quantity'] * df['price']

print(f'Merged dataset: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head()

## Step 6: Exploratory Data Analysis (EDA)

### 6a. Top 10 Customers by Revenue

In [ ]:
top_customers = (
    df.groupby(['customer_id', 'first_name', 'last_name'])
    .agg(
        total_revenue=('revenue', 'sum'),
        total_orders=('order_id', 'nunique'),
        avg_order_value=('revenue', 'mean'),
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
    .head(10)
)
top_customers

### 6b. Best-Selling Products

In [ ]:
best_products = (
    df.groupby(['product_name', 'category'])
    .agg(
        units_sold=('quantity', 'sum'),
        total_revenue=('revenue', 'sum'),
        avg_qty_per_order=('quantity', 'mean'),
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)
best_products.head(10)

### 6c. Revenue Overview

In [ ]:
print('=== Revenue Overview ===')
print(f'Total Revenue:     ${df["revenue"].sum():,.2f}')
print(f'Average Order:     ${df["revenue"].mean():,.2f}')
print(f'Median Order:      ${df["revenue"].median():,.2f}')
print(f'Total Orders:      {df["order_id"].nunique():,}')
print(f'Total Customers:   {df["customer_id"].nunique():,}')
print(f'Date Range:        {df["order_date"].min().date()} to {df["order_date"].max().date()}')

## Step 7: GroupBy Analysis

### Revenue by Category

In [ ]:
cat_revenue = (
    df.groupby('category')
    .agg(
        total_revenue=('revenue', 'sum'),
        pct_of_total=('revenue', lambda x: x.sum() / df['revenue'].sum() * 100),
        avg_order=('revenue', 'mean'),
        order_count=('order_id', 'count'),
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)
cat_revenue

### Revenue by Region

In [ ]:
region_revenue = (
    df.groupby('region')
    .agg(
        total_revenue=('revenue', 'sum'),
        customer_count=('customer_id', 'nunique'),
        avg_revenue_per_customer=('revenue', lambda x: x.sum() / df.loc[x.index, 'customer_id'].nunique()),
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)
region_revenue

### Monthly Revenue

In [ ]:
df['order_month'] = df['order_date'].dt.to_period('M')

monthly_revenue = (
    df.groupby('order_month')
    .agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'nunique'),
        avg_order_value=('revenue', 'mean'),
    )
    .round(2)
)
monthly_revenue

## Step 8: Time Series Analysis

In [ ]:
# Monthly revenue trend
monthly_ts = df.set_index('order_date')['revenue'].resample('ME').sum()

# Add moving averages
monthly_df = pd.DataFrame({
    'revenue': monthly_ts,
    'MA_3': monthly_ts.rolling(3).mean(),
    'MA_6': monthly_ts.rolling(6).mean(),
}).round(2)

monthly_df

In [ ]:
# Month-over-month growth
monthly_df['mom_growth'] = monthly_df['revenue'].pct_change() * 100
monthly_df['cumulative_revenue'] = monthly_df['revenue'].cumsum()

monthly_df.round(2)

In [ ]:
# Day-of-week analysis
df['day_of_week'] = df['order_date'].dt.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_analysis = (
    df.groupby('day_of_week')
    .agg(
        total_revenue=('revenue', 'sum'),
        avg_revenue=('revenue', 'mean'),
        order_count=('order_id', 'count'),
    )
    .reindex(day_order)
    .round(2)
)
dow_analysis

## Step 9: Pivot Tables & Cross-Tabulation

In [ ]:
# Revenue by Category × Region
pivot_cat_region = pd.pivot_table(
    df,
    values='revenue',
    index='category',
    columns='region',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(2)

pivot_cat_region

In [ ]:
# Order count cross-tabulation
order_crosstab = pd.crosstab(
    df['category'],
    df['region'],
    margins=True,
    margins_name='Total'
)
order_crosstab

In [ ]:
# Average order value by Category × Region
avg_order_pivot = pd.pivot_table(
    df,
    values='revenue',
    index='category',
    columns='region',
    aggfunc='mean',
    margins=True,
    margins_name='Average'
).round(2)

avg_order_pivot

## Step 10: Summary Report

In [ ]:
# Compile key findings
total_rev = df['revenue'].sum()
total_orders = df['order_id'].nunique()
total_customers = df['customer_id'].nunique()
avg_order_val = df['revenue'].mean()

top_category = cat_revenue.index[0]
top_cat_rev = cat_revenue.iloc[0]['total_revenue']

top_region = region_revenue.index[0]
top_region_rev = region_revenue.iloc[0]['total_revenue']

best_product_name = best_products.index[0][0]
best_product_rev = best_products.iloc[0]['total_revenue']

best_month = monthly_df['revenue'].idxmax()
best_month_rev = monthly_df['revenue'].max()

report = f"""
{'='*60}
       E-COMMERCE DATA ANALYSIS — SUMMARY REPORT
{'='*60}

📊 OVERVIEW
{'─'*40}
  Total Revenue:        ${total_rev:>12,.2f}
  Total Orders:         {total_orders:>12,}
  Unique Customers:     {total_customers:>12,}
  Avg Order Value:      ${avg_order_val:>12,.2f}

🏆 TOP PERFORMERS
{'─'*40}
  Best Category:        {top_category} (${top_cat_rev:,.2f})
  Best Region:          {top_region} (${top_region_rev:,.2f})
  Best Product:         {best_product_name} (${best_product_rev:,.2f})
  Best Month:           {best_month} (${best_month_rev:,.2f})

📈 REVENUE BY CATEGORY
{'─'*40}
"""

for cat, row in cat_revenue.iterrows():
    bar = '█' * int(row['pct_of_total'] / 2)
    report += f"  {cat:<15} ${row['total_revenue']:>10,.2f}  ({row['pct_of_total']:.1f}%) {bar}\n"

report += f"""
🌍 REVENUE BY REGION
{'─'*40}
"""

for reg, row in region_revenue.iterrows():
    report += f"  {reg:<15} ${row['total_revenue']:>10,.2f}  ({row['customer_count']} customers)\n"

report += f"""
{'='*60}
                    END OF REPORT
{'='*60}
"""

print(report)

## 🎉 Congratulations!

You've completed the entire Pandas course! Here's what you've learned:

| Chapter | Topic |
|---------|-------|
| 1 | Series & DataFrame fundamentals |
| 2 | Reading & writing data (CSV, JSON, Excel) |
| 3 | Selecting & filtering data |
| 4 | Data cleaning & transformation |
| 5 | GroupBy & aggregation |
| 6 | Merging, joining & concatenation |
| 7 | Reshaping data (pivot, melt, stack) |
| 8 | Time series analysis |
| 9 | Performance tips & best practices |
| 10 | Complete data analysis workflow |

### Next Steps

- Practice with real datasets (Kaggle, UCI ML Repository)
- Learn **Matplotlib** / **Seaborn** for data visualization
- Explore **Scikit-learn** for machine learning
- Try **Polars** for even faster data processing